# Regime-Switched Trading Strategy

This notebook is the orchestration layer for the integrated trading strategy.
The reusable logic lives in `project_dataset.py` and `regime_strategy.py`.

Current implementation:
- Predict the next-day regime with an HMM using an explicit train/test split.
- Use only one-step-ahead forecasts from `Next_Day_Forecast_Prob_High_Vol`.
- Switch regime only when the predicted probability of the other regime exceeds 90%.
- Use momentum on the small-cap universe in low volatility.
- Use mean reversion on the large-cap universe in high volatility.
- Close all positions at each regime change, then open the new regime portfolio.
- Apply a fixed transaction cost, with a sweep to study when the strategy becomes unprofitable.
- Keep the sentiment layer out of the notebook for now. It will be added later as an extension to the low-volatility leg.


# Required Datasets

This notebook assumes the usable datasets are already present under `data/`.
The current default configuration uses the `no_dividends` policy and loads the strategy universes from the per-ticker folders `data/large_caps/` and `data/small_caps/`.
The raw `vb_prices_volumes.csv` and `voo_prices_volumes.csv` files generated by `dataset_maker.py` are intentionally ignored here because they are not clean enough for direct use.

Detailed list of data currently required for this implementation:
- `data/large_caps/*.csv`: one CSV per large-cap ticker, each containing at least `Date`, `Close`, and `Volume`. The notebook currently uses the `Close` column for the high-volatility large-cap universe.
- `data/small_caps/*.csv`: one CSV per small-cap ticker, each containing at least `Date`, `Close`, and `Volume`. The notebook currently uses the `Close` column for the low-volatility small-cap universe.
- `data/portfolio_returns_equal_weight.csv`: equal-weight return series used to build the market-level HMM feature set. In this notebook, the market return can be the large-cap series, the small-cap series, or the average of both. For the current setup, the no-dividend columns are required.
- `data/vix_regime_classification.csv`: must contain `VIX_Close` for the HMM feature table.
- `data/spy_volume_features.csv`: must contain `SPY_Volume_Change`, used as an extra market feature.
- `data/universe_info.csv`: metadata only, but useful for validating the universe definition and later reporting.

Data that would be needed for a more production-grade implementation:
- Historical universe membership by date to avoid survivorship bias.
- A documented policy for delistings, ticker changes, and missing histories.
- Corporate-action-cleaned total-return data with a clear dividend treatment.
- Asset-level volumes if later strategies need liquidity filters or volume features.
- A daily sentiment signal aligned to trading dates for the low-volatility overlay.
- An execution convention, for example close-to-close or close-to-next-open, with transaction cost assumptions matched to that execution model.


In [12]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from IPython.display import display

import pandas as pd
import plotly.graph_objects as go

from project_dataset import load_regime_strategy_data
from regime_strategy import RegimeStrategyConfig
from regime_strategy_experiments import run_train_test_optimized_vix_strategy

pd.options.display.float_format = "{:,.6f}".format


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
data_dir = Path("data")
dividend_policy = "no_dividends"
universe_price_source = "folders"

# Cached portfolio-return series are still loaded.
# The active strategy below uses a VIX rule with train/test parameter optimization.
market_return_mode = "average"

# Optimization setup:
# - train on 1990-2009
# - test on 2010-2025
# - low-vol strategy is weekly buy-and-hold on all stocks
# - high-vol strategy is weekly mean reversion on all stocks
# - regime switches are rebalanced weekly using the last signal of each week
train_start_date = "2010-01-01"
split_date = "2017-01-01"
test_end_date = "2025-12-31"
rebalance_rule = "W-FRI"
optimization_metric = "Net_Sharpe"
parameter_grid = {
    "vix_lookback_days": [10, 20, 25],
    "vix_threshold_multiplier": [1.10, 1.15, 1.20],
    "mean_reversion_window": [3, 5, 10],
    "mean_reversion_theta": [0.5, 0.75, 1.0],
}

config = RegimeStrategyConfig(
    split_date=split_date,
    regime_probability_threshold=0.90,
    regime_min_days=None,
    transaction_cost_bps=10.0,
    realized_vol_window=20,
    momentum_fast_window=21,
    momentum_slow_window=252,
    mean_reversion_window=5,
    mean_reversion_theta=0.75,
    use_log_returns=False,
    liquidate_on_regime_change=True,
)

config


RegimeStrategyConfig(split_date='2017-01-01', regime_probability_threshold=0.9, regime_min_days=None, transaction_cost_bps=10.0, realized_vol_window=20, momentum_fast_window=21, momentum_slow_window=252, mean_reversion_window=5, mean_reversion_theta=0.75, use_log_returns=False, liquidate_on_regime_change=True)

In [14]:
strategy_data = load_regime_strategy_data(
    data_dir=data_dir,
    dividend_policy=dividend_policy,
    market_return_mode=market_return_mode,
    universe_price_source=universe_price_source,
    start_date=train_start_date,
    end_date=test_end_date,
)

print("Small-cap panel shape:", strategy_data.small_cap_prices.shape)
print("Large-cap panel shape:", strategy_data.large_cap_prices.shape)
print("Market return range:", strategy_data.market_return.dropna().index.min().date(), "to", strategy_data.market_return.dropna().index.max().date())
print("VIX range:", strategy_data.vix_close.dropna().index.min().date(), "to", strategy_data.vix_close.dropna().index.max().date())
strategy_data.metadata


Small-cap panel shape: (4024, 10)
Large-cap panel shape: (4024, 503)
Market return range: 2010-01-04 to 2025-12-31
VIX range: 2010-01-04 to 2025-12-31


{'data_dir': 'data',
 'dividend_policy': 'no_dividends',
 'market_return_mode': 'average',
 'universe_price_source': 'folders',
 'low_vol_universe': 'small_caps',
 'high_vol_universe': 'large_caps',
 'notes': ['The raw teammate-generated vb/voo prices_volumes files are intentionally ignored here.',
  "When universe_price_source='folders', the strategy universe prices come from data/large_caps and data/small_caps.",
  'The HMM market return still comes from data/portfolio_returns_equal_weight.csv.']}

In [15]:
result = run_train_test_optimized_vix_strategy(
    small_cap_price_panel=strategy_data.small_cap_prices,
    large_cap_price_panel=strategy_data.large_cap_prices,
    vix_close=strategy_data.vix_close,
    base_config=config,
    train_start_date=train_start_date,
    split_date=split_date,
    test_end_date=test_end_date,
    parameter_grid=parameter_grid,
    rebalance_rule=rebalance_rule,
    optimization_metric=optimization_metric,
)

print(
    "Training sample:",
    result["selected_parameters"]["Train_Start_Date"],
    "to",
    result["selected_parameters"]["Train_End_Date"],
    "| Test sample:",
    result["selected_parameters"]["Test_Start_Date"],
    "to",
    result["selected_parameters"]["Test_End_Date"],
)

display(result["selected_parameters"].rename("Selected_Value").to_frame())

display(result["optimization_table"][[
    "VIX_Lookback_Days",
    "VIX_Threshold_Multiplier",
    "Mean_Reversion_Window",
    "Mean_Reversion_Theta",
    "Optimization_Score",
    "Train_Net_Sharpe",
    "Train_Net_Annualized_Return",
    "Train_Net_Total_Return",
]].head(10))

display(pd.DataFrame(
    {
        "Train Strategy (Best Params)": result["train_summary"],
        "Test Strategy": result["summary"],
        "Test Benchmark: Long Low Vol / Short High Vol": result["all_stocks_long_short_benchmark_summary"],
    }
).T[[
    "Net_Total_Return",
    "Net_Annualized_Return",
    "Net_Sharpe",
    "Net_Max_Drawdown",
    "Average_Gross_Exposure",
    "Average_Net_Exposure",
]] )



Training sample: 2010-01-04 to 2016-12-31 | Test sample: 2017-01-01 to 2025-12-31


,Selected_Value
Optimization_Metric,Net_Sharpe
Requested_Train_Start_Date,2010-01-01
Train_Start_Date,2010-01-04
Train_End_Date,2016-12-31
Test_Start_Date,2017-01-01
Requested_Test_End_Date,2025-12-31
Test_End_Date,2025-12-31
VIX_Lookback_Days,20
VIX_Threshold_Multiplier,1.200000
Mean_Reversion_Window,10


,VIX_Lookback_Days,VIX_Threshold_Multiplier,Mean_Reversion_Window,Mean_Reversion_Theta,Optimization_Score,Train_Net_Sharpe,Train_Net_Annualized_Return,Train_Net_Total_Return
52,20,1.200000,10,0.750000,1.113840,1.113840,0.164383,1.856608
48,20,1.200000,5,0.500000,1.112570,1.112570,0.163516,1.841984
53,20,1.200000,10,1.000000,1.110792,1.110792,0.162893,1.831499
51,20,1.200000,10,0.500000,1.109274,1.109274,0.164433,1.857464
49,20,1.200000,5,0.750000,1.097793,1.097793,0.160792,1.796400
50,20,1.200000,5,1.000000,1.096490,1.096490,0.159024,1.767152
6,10,1.100000,10,0.500000,1.086073,1.086073,0.155074,1.718283
7,10,1.100000,10,0.750000,1.069230,1.069230,0.151774,1.664860
21,10,1.200000,5,0.500000,1.068801,1.068801,0.162865,1.848024
18,10,1.200000,3,0.500000,1.064355,1.064355,0.161539,1.825583


,Net_Total_Return,Net_Annualized_Return,Net_Sharpe,Net_Max_Drawdown,Average_Gross_Exposure,Average_Net_Exposure
Train Strategy (Best Params),1.856608,0.164383,1.113840,-0.156352,0.952282,0.929120
Test Strategy,1.944762,0.128100,0.808202,-0.241098,0.955470,0.935204
Test Benchmark: Long Low Vol / Short High Vol,1.370612,0.101122,0.526400,-0.210034,1.000000,0.818423


In [16]:
result["low_vol_strategy"]["summary"], result["high_vol_strategy"]["summary"]


(Strategy_Name         buy_hold
 Assets                     513
 Start_Date          2017-01-03
 End_Date            2025-12-31
 Rebalance_Rule           W-FRI
 Signal_Frequency        weekly
 dtype: object,
 Strategy_Name       mean_reversion
 Assets                         513
 Start_Date              2017-01-03
 End_Date                2025-12-31
 Rebalance_Rule               W-FRI
 Signal_Frequency            weekly
 dtype: object)

In [17]:
columns_to_show = [
    "VIX_Close",
    "VIX_Past_25D_Average",
    "VIX_High_Vol_Threshold",
    "VIX_Ratio_To_Past_Average",
    "Decision_Regime",
    "Live_Regime",
    "Active_Strategy",
    "Gross_Strategy_Return",
    "Transaction_Cost",
    "Net_Strategy_Return",
    "Turnover",
    "Gross_Exposure",
    "Net_Exposure",
]


In [18]:
all_stocks_strategy = result["backtest"]["Cum_Net_Strategy"].dropna()
all_stocks_long_short_benchmark = result["all_stocks_long_short_benchmark_backtest"]["Cum_Net_Strategy"].dropna()
buy_hold_all_stocks = result["backtest"]["Cum_Buy_Hold_All_Stocks"].dropna()
regime_series = result["backtest"]["Live_Regime"].dropna()

performance_fig = go.Figure()
performance_fig.add_trace(go.Scatter(x=all_stocks_strategy.index, y=all_stocks_strategy, mode="lines", name="Strategy: buy & hold low vol / mean reversion high vol", line=dict(color="#d62728", width=2.2)))
performance_fig.add_trace(go.Scatter(x=all_stocks_long_short_benchmark.index, y=all_stocks_long_short_benchmark, mode="lines", name="Benchmark: long all stocks low vol / short all stocks high vol", line=dict(color="#ff7f0e", width=2.0, dash="dash")))
performance_fig.add_trace(go.Scatter(x=buy_hold_all_stocks.index, y=buy_hold_all_stocks, mode="lines", name="Buy & hold all stocks", line=dict(color="#8c564b", width=1.8, dash="dash")))
performance_fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(size=10, color="rgba(46, 204, 113, 0.45)"), name="Low vol regime strip"))
performance_fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(size=10, color="rgba(231, 76, 60, 0.45)"), name="High vol regime strip"))

def add_regime_strip(figure, regimes):
    if regimes.empty:
        return
    interval_start = regimes.index[0]
    current_regime = regimes.iloc[0]
    for timestamp, regime in regimes.iloc[1:].items():
        if regime != current_regime:
            figure.add_shape(
                type="rect",
                x0=interval_start,
                x1=timestamp,
                xref="x",
                fillcolor="rgba(46, 204, 113, 0.45)" if current_regime == "low_vol" else "rgba(231, 76, 60, 0.45)",
                line_width=0,
                layer="below",
                y0=0.0,
                y1=0.05,
                yref="paper",
            )
            interval_start = timestamp
            current_regime = regime
    figure.add_shape(
        type="rect",
        x0=interval_start,
        x1=regimes.index[-1],
        xref="x",
        fillcolor="rgba(46, 204, 113, 0.45)" if current_regime == "low_vol" else "rgba(231, 76, 60, 0.45)",
        line_width=0,
        layer="below",
        y0=0.0,
        y1=0.05,
        yref="paper",
    )

add_regime_strip(performance_fig, regime_series)
performance_fig.update_layout(
    title="Optimized On 1990-2009, Tested On 2010-2025",
    xaxis_title="Date",
    yaxis_title="Growth of $1",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
)
performance_fig.show()

vix_plot = result["regime_decisions"][["VIX_Close", "VIX_Past_25D_Average", "VIX_High_Vol_Threshold"]].dropna()
vix_fig = go.Figure()
vix_fig.add_trace(go.Scatter(
    x=vix_plot.index,
    y=vix_plot["VIX_Close"],
    mode="lines",
    name="VIX close",
    line=dict(color="#d62728", width=2),
))
vix_fig.add_trace(go.Scatter(
    x=vix_plot.index,
    y=vix_plot["VIX_Past_25D_Average"],
    mode="lines",
    name="Past 25D VIX average",
    line=dict(color="#1f77b4", width=1.8),
))
vix_fig.add_trace(go.Scatter(
    x=vix_plot.index,
    y=vix_plot["VIX_High_Vol_Threshold"],
    mode="lines",
    name="High-vol threshold",
    line=dict(color="#ff7f0e", width=1.8, dash="dash"),
))
vix_fig.update_layout(
    title="VIX Regime Rule",
    xaxis_title="Date",
    yaxis_title="VIX level",
    template="plotly_white",
    hovermode="x unified",
)
vix_fig.show()

cost_sweep = result["transaction_cost_sweep"].copy()
cost_fig = go.Figure()
cost_fig.add_trace(go.Scatter(x=cost_sweep["Transaction_Cost_Bps"], y=cost_sweep["Net_Total_Return"], mode="lines+markers", name="Net total return", line=dict(color="#1f77b4", width=2)))
cost_fig.add_trace(go.Scatter(x=cost_sweep["Transaction_Cost_Bps"], y=cost_sweep["Net_Annualized_Return"], mode="lines+markers", name="Net annualized return", line=dict(color="#ff7f0e", width=2)))
cost_fig.add_hline(y=0.0, line_dash="dash", line_color="black")

annualized_break_even = result["break_even_costs"].get("Break_Even_Cost_Bps_Annualized_Return")
if pd.notna(annualized_break_even):
    cost_fig.add_vline(x=float(annualized_break_even), line_dash="dot", line_color="#ff7f0e", annotation_text="Annualized break-even", annotation_position="top right")

total_return_break_even = result["break_even_costs"].get("Break_Even_Cost_Bps_Total_Return")
if pd.notna(total_return_break_even):
    cost_fig.add_vline(x=float(total_return_break_even), line_dash="dot", line_color="#1f77b4", annotation_text="Total-return break-even", annotation_position="bottom right")

cost_fig.update_layout(
    title="Profitability vs Transaction Cost",
    xaxis_title="Transaction cost (bps)",
    yaxis_title="Return",
    template="plotly_white",
    hovermode="x unified",
)
cost_fig.show()
